### Here we will analyze the user journeys, defining what a journey is. If we want to analyze a listening of a tour, we should be taking acount a specific session of a user. A user might listen to a tour a different date so we need to be able to define when a user "journey" begins and ends to get specific info about this. 

### In the previous notebooks:
1) We analyzed the robustness of the pseudo user id finding that we will use that id for user authentication
2) We found the correct story order of the tours from which we can analyze how much the users follow the correct path when listening to a tour 
3) We found that IOS data cannot be used to provide clear insights without altering the data based on assumptions that are not 100% robust (We dont know for sure when a story begun and ended).

In [1]:
import gc
import pandas as pd
from pathlib import Path

In [ ]:
# del story_events
# gc.collect()

In [2]:
BASE_DIR = Path.cwd().parent
events_path = BASE_DIR / "data/clean/events.parquet"

In [3]:
events_data = pd.read_parquet(events_path)

In [4]:
listening_events = events_data.loc[~events_data.tour_id.isna()]

In [5]:
ANDROID = listening_events[listening_events['platform'] == "ANDROID"]

## 1) Defining user journeys.

lets check if the same user_pseudo_id listened to the same tour_id on multiple dates.

In [15]:
ANDROID.groupby("event_name", observed=True).size().sort_values(ascending=False)

event_name
tour_download_progress    1216412
story_start                696476
story_listened_20          489766
story_listened_40          444479
story_listened_60          412362
story_listened_80          385498
story_completed            350942
click_item                 263233
screen_view                232970
collapse_player            142253
pause                      120313
play                        67718
click_story                 39600
start_tour                  38195
forward_10                  33652
click_progress_bar          27353
backward_10                 25019
click_listen_now            22201
expand_player               21842
next_story                  10162
previous_story               6828
click_more                   6405
click_back                   5174
tour_download_started        4032
click_download_tour          4027
click_location               3630
tour_download_ended          3227
click_3d_map                 1831
click_copy_ref_code           890
cli

In [16]:
listening_events = [
    "start_tour",
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
    "play",
    "pause",
    "forward_10",
    "backward_10",
    "click_progress_bar",
    "next_story",
    "previous_story",
    "click_story",
    "click_item",
    "click_listen_now"
]

ANDROID_LISTEN = ANDROID[ANDROID["event_name"].isin(listening_events)]

In [17]:
df = ANDROID_LISTEN[["user_pseudo_id", "tour_id", "event_datetime"]].dropna(subset=["tour_id"]).copy()

df["listen_date"] = pd.to_datetime(df["event_datetime"]).dt.date

tour_days = (
    df.groupby(["user_pseudo_id", "tour_id"])["listen_date"]
      .nunique()
      .reset_index(name="distinct_days")
)

multi_day_tours = tour_days[tour_days["distinct_days"] > 1]

print("user-tour pairs total:", len(tour_days))
print("user-tour pairs on multiple days:", len(multi_day_tours))
print("percentage:", len(multi_day_tours) / len(tour_days))

multi_day_tours.head(20)

user-tour pairs total: 10121
user-tour pairs on multiple days: 2498
percentage: 0.24681355597272997


,user_pseudo_id,tour_id,distinct_days
3,000b52af1c5a69eb96f0ef158a18dbe0,535,2
7,003cde637b910a8e8c5e911db3ef41a6,820,2
10,0076ca9ffc64b175dae1cdad029124dd,107,2
12,00810b9e3c1156eddc1326d14cc6ca64,869,2
19,00bee13b161139ae8887bdc825a511e0,535,2
21,00c9f1dd8dc43f45b70bef081b0b8571,865,2
22,00d1ca3ccdc87db57b02f3925913f2b7,107,4
24,00d2c86db17217114ba59dac500cf34b,858,2
31,01099eb2970b85d4eb3cc6c34c628800,539,2
32,010cccb789f7ea4ebc32f50ceb9ed292,873,2


In [19]:
examples = multi_day_tours.head(20)[["user_pseudo_id", "tour_id"]]

example_rows = df.merge(examples, on=["user_pseudo_id", "tour_id"], how="inner")

example_rows = example_rows.sort_values(["user_pseudo_id", "tour_id", "event_datetime"])
example_rows[["user_pseudo_id", "tour_id", "event_datetime", "listen_date"]]

,user_pseudo_id,tour_id,event_datetime,listen_date
7240,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:15.358000,2025-10-19
7241,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:16.175002,2025-10-19
7242,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:32.041004,2025-10-19
7243,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:49:47.082005,2025-10-19
7244,000b52af1c5a69eb96f0ef158a18dbe0,535,2025-10-19 12:50:02.433006,2025-10-19
...,...,...,...,...
6549,02992e8666e21683bb2eefecb9aab742,473,2025-10-17 17:11:15.469039,2025-10-17
6550,02992e8666e21683bb2eefecb9aab742,473,2025-10-17 17:11:18.202040,2025-10-17
6551,02992e8666e21683bb2eefecb9aab742,473,2025-10-17 17:11:21.324042,2025-10-17
6552,02992e8666e21683bb2eefecb9aab742,473,2025-10-17 17:11:24.534000,2025-10-17


In [12]:
weird_guy = ANDROID[ANDROID['user_pseudo_id'] == "00bb0aa3cebec049772d02e9fba36394"]

In [14]:
ANDROID.loc[
    (ANDROID['tour_id'] == 107) &
    (ANDROID['language'] == "es-es") &
    (ANDROID['event_datetime'].dt.date == pd.to_datetime("2025-07-27").date())
]

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
463948,2025-07-27 07:37:20.017000,click_listen_now,ANDROID,es-es,<NA>,81e1d319ba42b04339ada0762300486f,107,<NA>,8,NaN,NaN
461670,2025-07-27 07:37:32.343002,start_tour,ANDROID,es-es,<NA>,81e1d319ba42b04339ada0762300486f,107,<NA>,8,NaN,NaN
462120,2025-07-27 07:37:33.313004,story_start,ANDROID,es-es,<NA>,81e1d319ba42b04339ada0762300486f,107,50606,8,NaN,NaN
1177042,2025-07-27 07:37:33.315005,collapse_player,ANDROID,es-es,<NA>,81e1d319ba42b04339ada0762300486f,107,50606,8,NaN,NaN
1179252,2025-07-27 07:49:06.659008,click_listen_now,ANDROID,es-es,<NA>,00bb0aa3cebec049772d02e9fba36394,107,<NA>,8,NaN,NaN
1177855,2025-07-27 07:49:13.847000,start_tour,ANDROID,es-es,<NA>,00bb0aa3cebec049772d02e9fba36394,107,<NA>,8,NaN,NaN
1447449,2025-07-27 07:49:14.197002,story_start,ANDROID,es-es,<NA>,00bb0aa3cebec049772d02e9fba36394,107,50606,8,NaN,NaN
910839,2025-07-27 07:49:14.202003,collapse_player,ANDROID,es-es,<NA>,00bb0aa3cebec049772d02e9fba36394,107,50606,8,NaN,NaN
909613,2025-07-27 07:49:29.205004,story_listened_20,ANDROID,es-es,<NA>,00bb0aa3cebec049772d02e9fba36394,107,50606,8,NaN,NaN
461763,2025-07-27 18:15:27.450008,start_tour,ANDROID,es-es,<NA>,9f3a52255cf0eb4be270ba34df99a456,107,<NA>,8,NaN,NaN


In [ ]:
events_data.loc[
    (events_data["event_name"] == 'start_tour') & (events_data["tour_id"].isna())
]